# PDF Metadata for ETU TORUS Source Document Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qrqx-ry0g/fair2.json`

The notebook follows a reusable exploration workflow:
- load Croissant metadata,
- inspect record sets and fields using their `@id` values,
- extract records into pandas DataFrames,
- apply simple processing steps,
- create a few exploratory visualizations.

> Important: throughout this notebook, dataset entities such as record sets and fields are referenced by their Croissant `@id` values whenever available.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the Croissant schema and inspect the dataset object.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qrqx-ry0g/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON for convenient display
metadata_json = dataset.metadata.to_json()

print("Dataset loaded successfully.")
print("Name:", metadata_json.get("name"))
print("Description:", metadata_json.get("description"))
print("Identifier:", metadata_json.get("identifier"))
print("Version:", metadata_json.get("version"))
print("License:", metadata_json.get("license"))

In [ ]:
# Show a compact metadata summary
summary_fields = [
    "name",
    "description",
    "identifier",
    "datePublished",
    "version",
    "license",
    "keywords",
    "dataCollection",
    "dataUseCases"
]

for field_name in summary_fields:
    value = metadata_json.get(field_name)
    print(f"\n{field_name}:")
    print(value)

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Some Croissant datasets expose one or more record sets for tabular access. This section discovers them dynamically from the schema JSON.

In [ ]:
# Inspect record sets directly from the JSON representation of the metadata
record_sets_json = metadata_json.get("recordSet", []) or []

print(f"Number of record sets declared in metadata: {len(record_sets_json)}")

record_set_overview = []
for rs in record_sets_json:
    rs_id = rs.get("@id", "<missing @id>")
    rs_name = rs.get("name", "<no name>")
    rs_desc = rs.get("description", "")
    field_list = rs.get("field", []) or []
    field_ids = [f.get("@id", "<missing @id>") for f in field_list]
    record_set_overview.append({
        "record_set_id": rs_id,
        "record_set_name": rs_name,
        "n_fields": len(field_list),
        "description": rs_desc,
        "field_ids": field_ids
    })

overview_df = pd.DataFrame(record_set_overview)
overview_df

In [ ]:
# Display fields for each record set using their @id values
for rs in record_sets_json:
    rs_id = rs.get("@id", "<missing @id>")
    rs_name = rs.get("name", "<no name>")
    print(f"\nRecord Set: {rs_name}")
    print(f"@id: {rs_id}")
    print("Fields:")
    
    for field in rs.get("field", []) or []:
        field_id = field.get("@id", "<missing @id>")
        field_name = field.get("name", "<no name>")
        field_desc = field.get("description", "")
        field_type = field.get("dataType", "<unknown>")
        print(f"  - {field_id} | name={field_name} | dataType={field_type}")
        if field_desc:
            print(f"    description={field_desc}")

If record sets are present, we can also preview a few records using the record set `@id`. If no record sets are declared, the later sections gracefully fall back to metadata-only exploration.

In [ ]:
# Preview records from the first available record set, if any
record_set_ids = [rs.get("@id") for rs in record_sets_json if rs.get("@id")]

if record_set_ids:
    preview_record_set_id = record_set_ids[0]
    print("Previewing records for record set @id:", preview_record_set_id)
    
    try:
        for i, record in enumerate(dataset.records(record_set=preview_record_set_id)):
            print(record)
            if i >= 4:
                break
    except Exception as e:
        print("Could not fetch records for this record set.")
        print("Reason:", e)
else:
    print("No record sets were declared in the metadata, so there are no tabular records to preview.")

## 3. Data Extraction
Load each record set into a pandas DataFrame using the record set `@id` values discovered above.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

print("\nLoaded DataFrame keys:")
print(list(dataframes.keys()))

In [ ]:
# Inspect columns and head of the first loaded DataFrame
if dataframes:
    example_record_set_id = next(iter(dataframes))
    example_df = dataframes[example_record_set_id]
    print("Example record set @id:", example_record_set_id)
    print("Columns:")
    print(example_df.columns.tolist())
    display(example_df.head())
else:
    print("No tabular record sets were loaded. This dataset appears to be metadata-centric in its current Croissant representation.")

## 4. Exploratory Data Analysis (EDA)
Apply simple processing steps to the extracted data when tabular record sets are available. The code below dynamically searches for numeric and grouping fields.

If the dataset does not expose record sets, the notebook instead performs EDA on metadata-derived information such as keywords, use cases, and bias statements.

In [ ]:
if dataframes:
    eda_record_set_id = next(iter(dataframes))
    eda_df = dataframes[eda_record_set_id].copy()
    
    # Try to identify numeric fields dynamically
    numeric_columns = eda_df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Attempt numeric conversion for object columns that may contain numbers
    if not numeric_columns:
        converted_df = eda_df.copy()
        for col in converted_df.columns:
            converted_df[col] = pd.to_numeric(converted_df[col], errors="ignore")
        eda_df = converted_df
        numeric_columns = eda_df.select_dtypes(include=[np.number]).columns.tolist()
    
    print("Numeric columns found:", numeric_columns)
    
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        threshold = eda_df[numeric_field_id].median()
        filtered_df = eda_df[eda_df[numeric_field_id] > threshold].copy()
        
        print(f"Using numeric field: {numeric_field_id}")
        print(f"Threshold (median): {threshold}")
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field_id].std()
        if std_value and not pd.isna(std_value) and std_value != 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / std_value
            )
        else:
            filtered_df[f"{numeric_field_id}_normalized"] = 0
        
        print(f"Normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        candidate_group_fields = [
            col for col in filtered_df.columns
            if filtered_df[col].dtype == "object" and filtered_df[col].nunique(dropna=True) > 0
        ]
        
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
            display(grouped_df.head())
        else:
            print("No suitable categorical grouping field was found.")
    else:
        print("No numeric fields were available for numeric EDA in the loaded record set.")
else:
    print("No record-set DataFrames are available, so we will perform metadata-based EDA instead.")
    
    keywords = metadata_json.get("keywords", []) or []
    use_cases = metadata_json.get("dataUseCases", []) or []
    biases = metadata_json.get("dataBiases", []) or []
    limitations = metadata_json.get("dataLimitations", []) or []
    
    metadata_eda = {
        "n_keywords": len(keywords),
        "n_use_cases": len(use_cases),
        "n_bias_statements": len(biases),
        "n_limitations": len(limitations),
        "description_length": len(str(metadata_json.get("description", ""))),
        "data_collection_length": len(str(metadata_json.get("dataCollection", "")))
    }
    
    metadata_eda_df = pd.DataFrame([metadata_eda])
    display(metadata_eda_df)
    
    print("\nKeywords:")
    for item in keywords:
        print("-", item)
    
    print("\nUse cases:")
    for item in use_cases:
        print("-", item)

## 5. Visualization
Visualize available data. When tabular records exist, this section plots numeric distributions and grouped summaries. Otherwise, it visualizes metadata-derived summaries.

In [ ]:
if dataframes:
    viz_record_set_id = next(iter(dataframes))
    viz_df = dataframes[viz_record_set_id].copy()
    
    for col in viz_df.columns:
        viz_df[col] = pd.to_numeric(viz_df[col], errors="ignore")
    
    numeric_columns = viz_df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        
        plt.figure(figsize=(8, 4))
        sns.histplot(viz_df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
        
        candidate_group_fields = [
            col for col in viz_df.columns
            if viz_df[col].dtype == "object" and viz_df[col].nunique(dropna=True) > 0 and viz_df[col].nunique(dropna=True) <= 20
        ]
        
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            grouped_plot_df = (
                viz_df.groupby(group_field_id)[numeric_field_id]
                .mean()
                .sort_values(ascending=False)
                .reset_index()
            )
            
            plt.figure(figsize=(10, 5))
            sns.barplot(data=grouped_plot_df, x=group_field_id, y=numeric_field_id)
            plt.title(f"Average {numeric_field_id} by {group_field_id}")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()
        else:
            print("No suitable categorical field found for grouped visualization.")
    else:
        print("No numeric columns available for visualization in the loaded record sets.")
else:
    keywords = metadata_json.get("keywords", []) or []
    use_cases = metadata_json.get("dataUseCases", []) or []
    biases = metadata_json.get("dataBiases", []) or []
    limitations = metadata_json.get("dataLimitations", []) or []
    
    counts_df = pd.DataFrame({
        "category": ["keywords", "use_cases", "bias_statements", "limitations"],
        "count": [len(keywords), len(use_cases), len(biases), len(limitations)]
    })
    
    plt.figure(figsize=(8, 4))
    sns.barplot(data=counts_df, x="category", y="count")
    plt.title("Metadata Summary Counts")
    plt.xlabel("Metadata category")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
    
    if keywords:
        keyword_lengths_df = pd.DataFrame({
            "keyword": keywords,
            "length": [len(k) for k in keywords]
        }).sort_values("length", ascending=False)
        
        plt.figure(figsize=(10, 5))
        sns.barplot(data=keyword_lengths_df, x="keyword", y="length")
        plt.title("Keyword Lengths")
        plt.xticks(rotation=60, ha="right")
        plt.xlabel("Keyword")
        plt.ylabel("Number of characters")
        plt.tight_layout()
        plt.show()
    else:
        print("No keywords available for metadata visualization.")

## 6. Conclusion
This notebook demonstrated how to:
- load a Croissant dataset with `mlcroissant`,
- inspect metadata safely through the dataset metadata object,
- discover record sets and fields using their `@id` values,
- extract records into pandas when tabular record sets are available,
- perform fallback metadata-based analysis when the dataset is primarily descriptive.

For this dataset, the Croissant package is strongly metadata-oriented and emphasizes provenance, bibliographic context, document status, and responsible use guidance. If future versions expose tabular `recordSet` entries, the same notebook structure can be reused without major changes.